# out-traveler 전처리 — jinkyeong (재구성본)

작성 2026-08-24 · 원본: `data/processed/travel_timing_expenditure_2025.csv` (14,342행 × 38열)
보조: `data/processed/accommodation_type_2025.csv` (19,803행)

이 노트북은 오늘 검증한 모든 규칙을 순서 꼬임 없이 처음부터 다시 정리한 최종본입니다.
각 단계는 **무엇을 / 왜**를 먼저 적고 코드를 붙였습니다. 원칙은 두 가지뿐입니다.

1. **애매한 값은 지우지 말고 플래그로 남긴다** (완전 중복·논리 모순처럼 명백히 틀린 값만 삭제 대상)
2. **판단 기준은 결과를 보기 전에 정한다** (사후조정 금지)


In [35]:
import pandas as pd
import numpy as np

df_exp = pd.read_csv("../../data/processed/travel_timing_expenditure_2025.csv")
acc = pd.read_csv("../../data/processed/accommodation_type_2025.csv")
df = df_exp.copy()
print("원본:", df.shape)

원본: (14342, 38)


## 1. x/y 중복 컬럼 통합

**왜**: `RESPOND_ID`+`EXAMIN_BEGIN_DE`로 두 원본 파일(여행시기/지출)을 병합하면서, 양쪽에 다 있던 컬럼(지역·인원·인적정보 등 11종)이 `_x`/`_y`로 중복 생성됐습니다. 100% 일치를 직접 확인했으므로(assert로 재확인) 하나로 합쳐서 컬럼을 정리합니다.

In [36]:
dup_pairs = ['EXAMIN_BEGIN_DE','TOUR_CTPRVN_NM','TOUR_SIGNGU_NM','TOUR_COM_NMPR_NM',
             'TOUR_PD_VALUE','SEXDSTN_FLAG_CD','AGRDE_FLAG_NM','MRRG_AT_NM',
             'CHLDRN_TY_NM','OCCP_NM','HSHLD_INCOME_DGREE_NM']

for c in dup_pairs:
    assert (df[c+'_x'] == df[c+'_y']).all(), f'{c} 불일치 발견 — 병합 재검토 필요'
    df = df.drop(columns=[c+'_y']).rename(columns={c+'_x': c})

print('통합 후:', df.shape)  # (14342, 27) 예상

통합 후: (14342, 27)


## 2. `'모름'` → 결측(NaN) + `_unk` 플래그

**왜**: 결측이 `NaN`이 아니라 문자열 `'모름'`으로 인코딩돼 있어 `isna()`로는 안 잡힙니다. 값은 결측 처리하되, "모름이었다"는 사실 자체를 플래그로 남깁니다 — 이 결측이 무작위가 아니라 연령·소득과 상관돼 있어서(20대 모름률이 50대의 2.5배), 그 자체가 분석 대상이기 때문입니다.

In [37]:
UNK = ['모름', '구체적인 지역 모름']
target_cols = [
    'TOUR_TOT_CT_VALUE','TOUR_LDGMNT_CT_VALUE','TOUR_FOOD_CT_VALUE',
    'TOUR_TRNSPORT_CT_VALUE','TOUR_SHOPNG_CT_VALUE','TOUR_ACTVTY_CT_VALUE','TOUR_ETC_CT_VALUE',
    'TOUR_PD_VALUE','TOUR_COM_NMPR_NM','HSHLD_INCOME_DGREE_NM','TOUR_SIGNGU_NM'
]
for col in target_cols:
    df[col + '_unk'] = df[col].isin(UNK)
    df[col] = df[col].where(~df[col].isin(UNK))

print('모름->NaN 후:', df.shape)  # (14342, 38) 예상

모름->NaN 후: (14342, 38)


## 3. 지출 결측이 블록결측(0 아니면 7)인지 검증

**왜**: "숙박비만 모른다" 같은 부분결측이 있다면 항목 합으로 총액을 채우는 게 가능합니다. 블록결측(0 아니면 7뿐)이면 그 방법 자체가 불가능하다는 뜻이고, 그래서 listwise deletion이 편의가 아니라 유일하게 가능한 선택이었다는 근거가 됩니다.

In [38]:
cost_cols = ['TOUR_TOT_CT_VALUE','TOUR_LDGMNT_CT_VALUE','TOUR_FOOD_CT_VALUE',
             'TOUR_TRNSPORT_CT_VALUE','TOUR_SHOPNG_CT_VALUE','TOUR_ACTVTY_CT_VALUE','TOUR_ETC_CT_VALUE']
unk_pattern = df[[c + '_unk' for c in cost_cols]].sum(axis=1)
print(unk_pattern.value_counts().sort_index())
assert set(unk_pattern.unique()) <= {0, 7}, "블록결측 가정이 깨짐 — 재검토 필요"

0    11703
7     2639
Name: count, dtype: int64


## 4. 숙박일수·동반인원 파생 (H3의 분모)

**왜**: "1인 1박당 지출"(H3 핵심 지표)의 분모입니다. `'6박 7일 이상'`·`'5명 이상'`은 열린 구간이라 하한값을 쓰면 지출밀도가 과대추정될 수 있어 `bound_assumed`로 남깁니다.

In [39]:
NIGHTS_MAP = {'1박 2일':1, '2박 3일':2, '3박 4일':3, '4박 5일':4, '5박 6일':5, '6박 7일 이상':6}
PARTY_MAP  = {'혼자서':1, '2명':2, '3명':3, '4명':4, '5명 이상':5}

df['nights'] = df['TOUR_PD_VALUE'].map(NIGHTS_MAP)
df['party_size'] = df['TOUR_COM_NMPR_NM'].map(PARTY_MAP)
df['bound_assumed'] = df['TOUR_PD_VALUE'].eq('6박 7일 이상') | df['TOUR_COM_NMPR_NM'].eq('5명 이상')

## 5. 총지출 중간값 대입 + 1인1박당 지출 + top-code 플래그

**왜**: 구간형이라 중간값 대입 없이는 연산이 안 됩니다. `45만원`은 base case 가정치일 뿐이고(45/50/60 민감도 비교 예정), 제주도 응답 66%가 최상단 구간에 몰려있어 이 가정치 하나에 평균이 크게 좌우됩니다 — `topcoded`로 남겨서 나중에 재계산 가능하게 합니다.

In [40]:
TOT_MID = {'10만원 미만':5, '10~20만원 미만':15, '20~30만원 미만':25,
           '30~40만원 미만':35, '40만원 이상':45}  # base case, 50/60은 추후 민감도 분석

df['tot_cost_mid'] = df['TOUR_TOT_CT_VALUE'].map(TOT_MID)
df['topcoded'] = df['TOUR_TOT_CT_VALUE'].eq('40만원 이상')
df['cost_per_person_night'] = df['tot_cost_mid'] / (df['nights'] * df['party_size'])

## 6. 1인1박당 지출의 IQR 초과값 — 삭제 아님, 플래그만

**왜**: 초과 건의 실체는 "1박 2일 혼자 여행"처럼 분모가 작아서 생긴 정상 값입니다. 삭제하면 실제 소비 패턴을 지우는 셈이라 플래그만 남기고, 지역 비교 시 평균·중앙값을 같이 봅니다.

In [41]:
q1, q3 = df['cost_per_person_night'].quantile([.25, .75])
upper = q3 + 1.5 * (q3 - q1)
df['ppn_high'] = df['cost_per_person_night'] > upper

## 7. 논리 정합성 검사 (구간 경계 기준)

**왜**: 총액 구간과 항목 합 구간이 서로 모순되는지 확인합니다. 중간값이 아니라 상한·하한으로 검사해야 대입 오차를 모순으로 오인하지 않습니다. 위반 0건이면 데이터 신뢰도 근거로 씁니다.

In [42]:
TOT_BOUNDS = {'10만원 미만':(0,10), '10~20만원 미만':(10,20), '20~30만원 미만':(20,30),
              '30~40만원 미만':(30,40), '40만원 이상':(40, np.inf)}
ITEM_BOUNDS = {'1만원 미만':(0,1), '1~3만원 미만':(1,3), '3~5만원 미만':(3,5),
               '5~7만원 미만':(5,7), '7~10만원 미만':(7,10), '10만원 이상':(10, np.inf)}
item_cols = ['TOUR_LDGMNT_CT_VALUE','TOUR_FOOD_CT_VALUE','TOUR_TRNSPORT_CT_VALUE',
             'TOUR_SHOPNG_CT_VALUE','TOUR_ACTVTY_CT_VALUE','TOUR_ETC_CT_VALUE']

full = df[df[cost_cols].notna().all(axis=1)]
item_lo = sum(full[c].map(lambda v: ITEM_BOUNDS[v][0]) for c in item_cols)
item_hi = sum(full[c].map(lambda v: ITEM_BOUNDS[v][1]) for c in item_cols)
tot_lo  = full['TOUR_TOT_CT_VALUE'].map(lambda v: TOT_BOUNDS[v][0])
tot_hi  = full['TOUR_TOT_CT_VALUE'].map(lambda v: TOT_BOUNDS[v][1])
violation = (tot_hi < item_lo) | (tot_lo > item_hi)
print('논리 모순 건수:', violation.sum())

논리 모순 건수: 0


## 8. `id_suspect` — 같은 RESPOND_ID인데 성별이 뒤바뀐 경우

**왜**: 소득·직업·혼인상태 등은 1년 안에 바뀔 수 있어 정상이지만, **성별은 안 바뀝니다.** 34개 ID(81행)에서 성별이 뒤바뀌는 걸 발견했고, 이는 `RESPOND_ID`가 항상 같은 한 사람을 가리키지 않을 수 있다는 뜻입니다.

**판단 기준(사전 등록)**: H1~H3는 성별·나이를 안 쓰므로 영향 없음 → 그대로 사용. **H4(인적정보 통제 회귀)에서만** `id_suspect` 포함/제외 버전을 둘 다 돌려서 계수가 바뀌는지 확인 — 결과를 본 뒤 빼는 게 아니라, 처음부터 두 버전을 비교하는 게 규칙.

In [43]:
sex_flip_ids = df.groupby('RESPOND_ID')['SEXDSTN_FLAG_CD'].transform('nunique') > 1
df['id_suspect'] = sex_flip_ids
print('id_suspect 행 수:', df['id_suspect'].sum())

id_suspect 행 수: 81


## 9. `trip_dup` — 같은 사람·같은 여행일·같은 시도가 중복 응답된 경우

**왜**: 78개 그룹(160행)에서 발견. 목적지·숙박일수·인원 같은 "하드 팩트"는 81~92% 일치하는데 목적·지출액 같은 "소프트 팩트"는 50%만 일치 — 실제로는 같은 여행이 회상지연 겹치는 두 조사 주차에서 중복 응답된 것으로 보입니다(우연히 같은 날 같은 도로 떠난 별개 여행일 확률은 매우 낮음).

**판단 기준(사전 등록)**: 삭제하지 않고 `trip_dup_keep` 플래그만 생성. 대표값은 **회상지연이 짧은(조사일이 여행일에 가까운) 응답**을 채택 — 이미 확인된 "회상지연 짧을수록 응답이 정확하다"는 근거를 그대로 적용. H1 계산 시 포함/제외 버전을 비교.

In [44]:
key_cols = ['RESPOND_ID','TOUR_BEGIN_DE','TOUR_CTPRVN_NM']
df['trip_dup_group'] = df.duplicated(subset=key_cols, keep=False)
df['trip_dup_rank'] = df.groupby(key_cols)['EXAMIN_BEGIN_DE'].rank(method='first')
df['trip_dup_keep'] = (~df['trip_dup_group']) | (df['trip_dup_rank'] == 1)

print('중복 그룹 행:', df['trip_dup_group'].sum(), '/ 제외 후보:', (~df['trip_dup_keep']).sum())

중복 그룹 행: 160 / 제외 후보: 82


## 10. 시군구 다목적지 분리 (별도 파일)

**왜**: `'가평군/화성시'`처럼 응답 하나가 시군구 두 곳을 가리키는 경우, 1/n 가중치로 행을 쪼개야 시군구 집계가 정확해집니다. 그런데 쪼개면 "1행=여행 1건"이라는 단위가 깨지므로, 메인 파일과 분리된 별도 파일로 저장합니다.

In [45]:
sig = df[['TOUR_CTPRVN_NM', 'TOUR_SIGNGU_NM']].reset_index().rename(columns={'index': 'row_id'})
sig = sig.dropna(subset=['TOUR_SIGNGU_NM'])
sig['TOUR_SIGNGU_NM'] = sig['TOUR_SIGNGU_NM'].str.split('/')
sig = sig.explode('TOUR_SIGNGU_NM')
sig['weight'] = 1 / sig.groupby('row_id')['TOUR_SIGNGU_NM'].transform('count')

print('sig:', sig.shape)  # (14532, 4) 예상

sig: (14532, 4)


## 11. `accommodation_type_2025.csv` 병합 — 응답자 단위 배경정보

**왜**: 이 파일은 여행 1건이 아니라 **응답자×월** 단위입니다(목적지 정보 없음). `DMSTC_TOUR_TY_VALUE`(개별/패키지 여행)가 같은 사람 안에서도 달마다 29% 확률로 바뀌어서, 특정 여행에 직접 라벨을 붙일 수 없습니다. 대신 **사람 단위 요약**(연중 개별여행 응답 비율, 대표 거주지)만 `RESPOND_ID`로 조인합니다.

메인 응답자의 99.3%가 이 파일에도 있어 조인은 잘 되지만, 59행(0.41%)은 매칭이 안 됩니다 — 이건 `'모름'`과 다른 성격의 결측(이 설문 자체에 응답한 적이 없음)이라 별도 플래그로 구분합니다.

In [46]:
person_summary = acc.groupby('RESPOND_ID').agg(
    indiv_travel_ratio=('DMSTC_TOUR_TY_VALUE', lambda x: (x == '개별 여행').mean()),
    residence=('ANSWRR_OC_AREA_NM', lambda x: x.mode()[0])
).reset_index()

df = df.merge(person_summary, on='RESPOND_ID', how='left')
df['acc_unmatched'] = df['indiv_travel_ratio'].isna()

print('병합 후:', df.shape, '/ acc_unmatched:', df['acc_unmatched'].sum())

병합 후: (14342, 52) / acc_unmatched: 59


## 12. 최종 저장

In [47]:
import os
os.makedirs('../../data/cleaned', exist_ok=True)

df.to_csv('../../data/cleaned/jinkyeong_cleaned.csv', index=False)
sig.to_csv('../../data/cleaned/jinkyeong_signgu_long.csv', index=False)

print('저장 완료:', df.shape, sig.shape)

저장 완료: (14342, 52) (14532, 4)


## 요약 — 오늘 반영된 사전 등록 규칙

| 발견 | 처리 | 적용 범위 |
|---|---|---|
| `'모름'` 결측 (연령·소득 편향) | 플래그(`_unk`) + 유지 | 전체 |
| 지출 블록결측 0/7 | listwise deletion 정당화 | 지출 분석 |
| 개방구간(6박7일↑, 5명↑, 40만원↑) | 플래그(`bound_assumed`, `topcoded`) + 유지 | 전체, 민감도 분석 예정 |
| 1인1박당 IQR 초과 | 플래그(`ppn_high`) + 유지 | 전체, 평균·중앙값 병기 |
| 성별 불일치 ID (34개/81행) | 플래그(`id_suspect`) + 유지 | **H4에서만** 포함/제외 비교 |
| 여행 중복응답 의심 (78그룹/160행) | 플래그(`trip_dup_keep`) + 유지, 회상지연 짧은 쪽 대표 | **H1에서** 포함/제외 비교 |
| accommodation 미매칭 (59행) | 플래그(`acc_unmatched`) + 유지 | H4 보조변수 사용 시 |

**아직 안 한 것**: top-code 민감도(45/50/60), `ppn_high`/`bound_assumed`/`id_suspect`/`trip_dup` 포함·제외 비교 실행 (H1~H4 단계에서 진행).